# 🌦️ Varsha Weather — Complete Model Training
### Indore City Weather Prediction System
---
**Models trained in this notebook:**

| # | Model | Target | Algorithm | Expected R² / Acc |
|---|-------|--------|-----------|-------------------|
| 1 | Temperature | `temperature_2m` (°C) | Random Forest Regressor | R² ≈ 0.997 |
| 2 | Humidity | `relative_humidity_2m` (%) | Random Forest Regressor | R² ≈ 0.998 |
| 3 | Wind Speed | `wind_speed_10m` (km/h) | Random Forest Regressor | R² ≈ 0.917 |
| 4 | Rain Amount | `rain` (mm) | Gradient Boosting Regressor | R² ≈ 0.30 |
| 5 | Rain Classifier | `will_rain` (0/1) | Random Forest Classifier | Acc ≈ 92% |

**Output:** `./models/` folder with 7 `.joblib` files ready for the Flask app.

---
## 📦 Section 1 — Install & Import Libraries

In [ ]:
# Uncomment the line below if packages are not installed
# !pip install pandas scikit-learn numpy joblib matplotlib seaborn

In [ ]:
import os
import math
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier,
    GradientBoostingRegressor,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

print("✅ All libraries imported successfully")

---
## ⚙️ Section 2 — Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────
DATA_PATH   = "Indorecity.csv"    # <-- change if your CSV is elsewhere
MODEL_DIR   = "./models"

# ── Training settings ──────────────────────────────────────────
SAMPLE_SIZE = 40_000   # rows to use (set None to use full dataset)
TEST_SIZE   = 0.20     # 20% held out for evaluation
RANDOM_SEED = 42

os.makedirs(MODEL_DIR, exist_ok=True)
print(f"✅ Config ready  |  Model output dir: {os.path.abspath(MODEL_DIR)}")

---
## 📂 Section 3 — Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape  : {df.shape[0]:,} rows  ×  {df.shape[1]} columns")
print(f"Columns        : {list(df.columns)}")
df.head()

In [ ]:
# Sample for faster training
if SAMPLE_SIZE and SAMPLE_SIZE < len(df):
    df = df.sample(SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Working sample : {len(df):,} rows")

print("\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string() or "  None")

In [ ]:
# Quick overview of key weather columns
target_cols = ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "rain"]
existing    = [c for c in target_cols if c in df.columns]
df[existing].describe().round(2)

---
## 🔧 Section 4 — Feature Engineering

We extract **cyclical time encodings** from the timestamp so the model understands that hour 23 and hour 0 are close together, and December and January are adjacent months.

In [ ]:
# Parse datetime
df["date"]        = pd.to_datetime(df["date"], utc=True)
df["hour"]        = df["date"].dt.hour
df["month"]       = df["date"].dt.month
df["day_of_year"] = df["date"].dt.dayofyear
df["day_of_week"] = df["date"].dt.dayofweek

# Cyclical sine/cosine encodings
df["hour_sin"]  = np.sin(2 * np.pi * df["hour"]        / 24)
df["hour_cos"]  = np.cos(2 * np.pi * df["hour"]        / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"]       / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"]       / 12)
df["doy_sin"]   = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["doy_cos"]   = np.cos(2 * np.pi * df["day_of_year"] / 365)

# Binary rain flag for classifier
df["will_rain"] = (df["rain"] > 0).astype(int)

print(f"✅ Features engineered")
print(f"   Rain events : {df['will_rain'].sum():,} / {len(df):,}  ({df['will_rain'].mean()*100:.1f}%)")

In [ ]:
# Define the 18 input features
FEATURE_CANDIDATES = [
    "hour_sin",  "hour_cos",
    "month_sin", "month_cos",
    "doy_sin",   "doy_cos",
    "day_of_week",
    "pressure_msl",        "surface_pressure",
    "cloud_cover",         "cloud_cover_low",
    "cloud_cover_mid",     "cloud_cover_high",
    "dew_point_2m",        "apparent_temperature",
    "wind_direction_10m",  "wind_gusts_10m",
    "snow_depth",
]

# Keep only columns that exist in this dataset
FEATURES = [f for f in FEATURE_CANDIDATES if f in df.columns]
MISSING  = [f for f in FEATURE_CANDIDATES if f not in df.columns]

print(f"✅ Features used ({len(FEATURES)}): {FEATURES}")
if MISSING:
    print(f"⚠️  Missing (skipped): {MISSING}")

X = df[FEATURES].fillna(0)
print(f"\nX shape: {X.shape}")

In [ ]:
# Visualise correlations between features and temperature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation heatmap (top features)
corr_cols = FEATURES + ["temperature_2m", "relative_humidity_2m"]
corr_cols = [c for c in corr_cols if c in df.columns]
corr = df[corr_cols].corr()[["temperature_2m", "relative_humidity_2m"]].drop(
    ["temperature_2m", "relative_humidity_2m"], errors="ignore"
)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", ax=axes[0], linewidths=0.5)
axes[0].set_title("Feature Correlations with Targets", fontweight="bold")

# Monthly temperature distribution
if "temperature_2m" in df.columns:
    df.groupby("month")["temperature_2m"].mean().plot(
        kind="bar", ax=axes[1], color="tomato", edgecolor="white"
    )
    axes[1].set_title("Mean Temperature by Month — Indore", fontweight="bold")
    axes[1].set_xlabel("Month")
    axes[1].set_ylabel("Temperature (°C)")
    axes[1].set_xticklabels(
        ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"],
        rotation=45
    )

plt.tight_layout()
plt.show()

---
## 🤖 Section 5 — Train All 5 Models

We train each model separately and collect results for the final summary.

In [ ]:
# Dictionary to store results
RESULTS = {}

def evaluate_regression(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    mae    = mean_absolute_error(y_test, y_pred)
    rmse   = math.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)
    print(f"   MAE  = {mae:.4f}")
    print(f"   RMSE = {rmse:.4f}")
    print(f"   R²   = {r2:.4f}")
    RESULTS[name] = {"type": "regression", "MAE": round(mae,4), "RMSE": round(rmse,4), "R2": round(r2,4)}
    return y_pred

def evaluate_classification(name, model, X_test, y_test):
    y_pred      = model.predict(X_test)
    y_prob      = model.predict_proba(X_test)[:, 1]
    acc         = accuracy_score(y_test, y_pred)
    auc         = roc_auc_score(y_test, y_prob)
    print(f"   Accuracy = {acc:.4f}")
    print(f"   ROC-AUC  = {auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=["No Rain","Rain"], zero_division=0))
    RESULTS[name] = {"type": "classification", "Accuracy": round(acc,4), "AUC": round(auc,4)}
    return y_pred

print("✅ Helper functions defined")

### 🌡️ Model 1 — Temperature (`temperature_2m`)

In [ ]:
print("=" * 50)
print("  MODEL 1: Temperature (°C)")
print("=" * 50)

y_temp = df["temperature_2m"].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_temp, test_size=TEST_SIZE, random_state=RANDOM_SEED
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Target range: {y_temp.min():.1f}°C → {y_temp.max():.1f}°C")

model_temp = RandomForestRegressor(
    n_estimators=100, max_depth=15,
    min_samples_split=4, n_jobs=-1, random_state=RANDOM_SEED
)
model_temp.fit(X_train, y_train)
pred_temp = evaluate_regression("temperature_2m", model_temp, X_test, y_test)

joblib.dump(model_temp, os.path.join(MODEL_DIR, "temperature_2m_model.joblib"))
print(f"\n💾 Saved  temperature_2m_model.joblib")

### 💧 Model 2 — Humidity (`relative_humidity_2m`)

In [ ]:
print("=" * 50)
print("  MODEL 2: Humidity (%)")
print("=" * 50)

y_humid = df["relative_humidity_2m"].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_humid, test_size=TEST_SIZE, random_state=RANDOM_SEED
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Target range: {y_humid.min():.1f}% → {y_humid.max():.1f}%")

model_humid = RandomForestRegressor(
    n_estimators=100, max_depth=15,
    min_samples_split=4, n_jobs=-1, random_state=RANDOM_SEED
)
model_humid.fit(X_train, y_train)
pred_humid = evaluate_regression("relative_humidity_2m", model_humid, X_test, y_test)

joblib.dump(model_humid, os.path.join(MODEL_DIR, "relative_humidity_2m_model.joblib"))
print(f"\n💾 Saved  relative_humidity_2m_model.joblib")

### 💨 Model 3 — Wind Speed (`wind_speed_10m`)

In [ ]:
print("=" * 50)
print("  MODEL 3: Wind Speed (km/h)")
print("=" * 50)

y_wind = df["wind_speed_10m"].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_wind, test_size=TEST_SIZE, random_state=RANDOM_SEED
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Target range: {y_wind.min():.1f} → {y_wind.max():.1f} km/h")

model_wind = RandomForestRegressor(
    n_estimators=100, max_depth=15,
    min_samples_split=4, n_jobs=-1, random_state=RANDOM_SEED
)
model_wind.fit(X_train, y_train)
pred_wind = evaluate_regression("wind_speed_10m", model_wind, X_test, y_test)

joblib.dump(model_wind, os.path.join(MODEL_DIR, "wind_speed_10m_model.joblib"))
print(f"\n💾 Saved  wind_speed_10m_model.joblib")

### 🌧️ Model 4 — Rain Amount (`rain`)
> **Note:** Rain is inherently chaotic and sparse. R² < 0.4 is expected and normal.

In [ ]:
print("=" * 50)
print("  MODEL 4: Rain Amount (mm)")
print("=" * 50)

y_rain = df["rain"].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_rain, test_size=TEST_SIZE, random_state=RANDOM_SEED
)
rain_pct = (y_rain > 0).mean() * 100
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Rain events: {rain_pct:.1f}%  |  Max: {y_rain.max():.2f}mm")

model_rain = GradientBoostingRegressor(
    n_estimators=100, max_depth=5,
    learning_rate=0.1, subsample=0.8, random_state=RANDOM_SEED
)
model_rain.fit(X_train, y_train)
pred_rain = evaluate_regression("rain", model_rain, X_test, y_test)

joblib.dump(model_rain, os.path.join(MODEL_DIR, "rain_model.joblib"))
print(f"\n💾 Saved  rain_model.joblib")

### ☔ Model 5 — Will It Rain? Classifier (`will_rain`)

In [ ]:
print("=" * 50)
print("  MODEL 5: Will-Rain Classifier (0/1)")
print("=" * 50)

y_cls = df["will_rain"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cls, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_cls
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Class balance — No Rain: {(y_cls==0).sum():,}  Rain: {(y_cls==1).sum():,}")

model_cls = RandomForestClassifier(
    n_estimators=100, max_depth=15,
    min_samples_split=4, n_jobs=-1,
    class_weight="balanced", random_state=RANDOM_SEED
)
model_cls.fit(X_train, y_train)
pred_cls = evaluate_classification("will_rain", model_cls, X_test, y_test)

joblib.dump(model_cls, os.path.join(MODEL_DIR, "will_rain_model.joblib"))
print(f"💾 Saved  will_rain_model.joblib")

---
## 💾 Section 6 — Save Metadata Files

In [ ]:
# Save ordered feature list (Flask app reads this at startup)
joblib.dump(FEATURES, os.path.join(MODEL_DIR, "feature_cols.joblib"))
print("💾 Saved  feature_cols.joblib")
print(f"   Features ({len(FEATURES)}): {FEATURES}")

In [ ]:
# Save descriptive statistics of all target columns
STAT_COLS = [c for c in [
    "temperature_2m", "relative_humidity_2m", "wind_speed_10m",
    "rain", "pressure_msl", "dew_point_2m",
    "cloud_cover", "wind_gusts_10m", "apparent_temperature",
] if c in df.columns]

joblib.dump(df[STAT_COLS].describe().to_dict(),
            os.path.join(MODEL_DIR, "data_stats.joblib"))
print("💾 Saved  data_stats.joblib")
df[STAT_COLS].describe().round(2)

---
## 📊 Section 7 — Results Summary & Visualisation

In [ ]:
print("=" * 55)
print("  TRAINING COMPLETE — Final Results")
print("=" * 55)
print(f"\n  {'Model':<28}  {'Metric':<10}  Value")
print("  " + "-" * 48)
for name, r in RESULTS.items():
    if r["type"] == "regression":
        print(f"  {name:<28}  {'R²':<10}  {r['R2']:.4f}   MAE={r['MAE']:.4f}")
    else:
        print(f"  {name:<28}  {'Accuracy':<10}  {r['Accuracy']:.4f}   AUC={r['AUC']:.4f}")
print()

In [ ]:
# Bar chart of model performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regression R² scores
reg_names  = [n for n,r in RESULTS.items() if r["type"]=="regression"]
reg_r2     = [RESULTS[n]["R2"] for n in reg_names]
short_names = ["Temperature", "Humidity", "Wind Speed", "Rain Amount"]

bars = axes[0].barh(short_names, reg_r2, color=["#e05c5c","#5b9bd5","#70ad47","#ffc000"],
                    edgecolor="white", height=0.5)
axes[0].set_xlim(0, 1.05)
axes[0].set_xlabel("R² Score", fontsize=12)
axes[0].set_title("Regression Models — R² Score\n(higher is better)", fontweight="bold")
for bar, val in zip(bars, reg_r2):
    axes[0].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{val:.3f}", va="center", fontsize=11)

# Classifier confusion matrix
y_cls_test = df["will_rain"]
_, X_te, _, y_te = train_test_split(X, y_cls_test, test_size=TEST_SIZE,
                                     random_state=RANDOM_SEED, stratify=y_cls_test)
cm = confusion_matrix(y_te, model_cls.predict(X_te))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
            xticklabels=["No Rain","Rain"], yticklabels=["No Rain","Rain"],
            linewidths=1, linecolor="white")
axes[1].set_title(f"Rain Classifier — Confusion Matrix\nAccuracy={RESULTS['will_rain']['Accuracy']:.3f}  AUC={RESULTS['will_rain']['AUC']:.3f}", fontweight="bold")
axes[1].set_ylabel("Actual")
axes[1].set_xlabel("Predicted")

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance plot (Temperature model — most important)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, model, title in [
    (axes[0], model_temp, "Temperature Model"),
    (axes[1], model_cls,  "Rain Classifier"),
]:
    fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)
    fi.tail(12).plot(kind="barh", ax=ax, color="#5b9bd5", edgecolor="white")
    ax.set_title(f"Feature Importance\n{title}", fontweight="bold")
    ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

---
## ✅ Section 8 — Verify Saved Files

In [ ]:
REQUIRED = [
    "temperature_2m_model.joblib",
    "relative_humidity_2m_model.joblib",
    "wind_speed_10m_model.joblib",
    "rain_model.joblib",
    "will_rain_model.joblib",
    "feature_cols.joblib",
    "data_stats.joblib",
]

print(f"Files in {os.path.abspath(MODEL_DIR)}/\n")
print(f"  {'File':<45}  {'Size':>8}  Status")
print("  " + "-" * 62)

all_ok = True
for fname in REQUIRED:
    path = os.path.join(MODEL_DIR, fname)
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1024 / 1024
        print(f"  ✅  {fname:<43}  {mb:>6.1f} MB")
    else:
        print(f"  ❌  {fname:<43}  MISSING")
        all_ok = False

print()
if all_ok:
    print("🎉 All 7 files present! Copy ./models/ into varsha_weather/ and run: python app.py")
else:
    print("⚠️  Some files are missing. Re-run the relevant training cell above.")

---
## 🚀 Next Steps

```
1. Copy ./models/ folder into varsha_weather/
2. Install Flask dependencies:
       pip install -r requirements.txt
3. Start the server:
       python app.py
4. Open browser:
       http://localhost:5000
```

**Project structure after training:**
```
varsha_weather/
├── app.py
├── models/
│   ├── temperature_2m_model.joblib      ← trained here
│   ├── relative_humidity_2m_model.joblib
│   ├── wind_speed_10m_model.joblib
│   ├── rain_model.joblib
│   ├── will_rain_model.joblib
│   ├── feature_cols.joblib
│   └── data_stats.joblib
├── templates/index.html
└── static/
    ├── css/style.css
    └── js/main.js
```